### Notebook 02 — Neural Bigram

#### Introdução

No notebook anterior construímos um modelo de linguagem baseado em contagens.

Ele aprendia observando quantas vezes um token aparecia após outro.

Por exemplo:

- o → l : 5 vezes
- o → espaço : 3 vezes

A partir dessas contagens calculávamos probabilidades.

Esse modelo funciona, mas possui limitações importantes.

Ele apenas memoriza o que observou durante o treinamento.

Neste notebook construiremos uma versão neural do mesmo modelo.

Ao invés de armazenar contagens explicitamente, utilizaremos parâmetros treináveis capazes de aprender essas relações automaticamente.

Essa mudança introduz vários conceitos fundamentais do Deep Learning:

- Pesos
- Matrizes
- Logits
- Softmax
- Loss
- Gradientes
- Backpropagation

Ao final deste notebook você terá treinado sua primeira rede neural para prever o próximo token.

### 1. Revisando o Modelo Anterior

#### Objetivo

Antes de avançarmos, vamos revisar rapidamente como funcionava nosso primeiro modelo.

O Bigram Language Model clássico aprendia observando pares de tokens consecutivos.

Exemplo:

Texto:

"ola"

Bigramas:

(o, l)

(l, a)

---

#### Como ele fazia previsões?

Depois de contar todas as ocorrências do dataset, o modelo construía uma tabela de probabilidades.

Exemplo:

Depois de "o":

- l → 75%
- espaço → 25%

Quando precisava gerar texto, ele consultava essa tabela e escolhia uma das opções disponíveis.

---

#### Fluxo completo

Texto
↓
Tokenização
↓
Contagem de Bigramas
↓
Probabilidades
↓
Geração de Texto

In [3]:
# ==================================================
# SEÇÃO 1 - REVISÃO
# ==================================================

print("Fluxo do modelo anterior:")

print("""
Texto
 ↓
Tokenização
 ↓
Contagem
 ↓
Probabilidades
 ↓
Geração
""")

Fluxo do modelo anterior:

Texto
 ↓
Tokenização
 ↓
Contagem
 ↓
Probabilidades
 ↓
Geração



#### Exercício de Reflexão

Imagine que o dataset contenha apenas:

"bom dia"

e

"boa noite"

Pergunta:

Após observar a letra "b", o modelo sabe qual frase está sendo construída?

Por quê?

#### Conceito Importante

O modelo clássico não possui compreensão.

Ele apenas consulta frequências observadas anteriormente.

Seu conhecimento está armazenado diretamente nas contagens.

### 2. O Problema do Bigrama Clássico

#### Objetivo

Entender por que precisamos de uma abordagem diferente.

O modelo baseado em contagens funciona bem para exemplos pequenos.

Mas apresenta limitações importantes quando o vocabulário cresce.

---

#### Imagine um vocabulário com 50.000 tokens

Para cada token precisaríamos armazenar informações sobre todos os possíveis próximos tokens.

O número de combinações cresce rapidamente.

Além disso, o modelo apenas memoriza frequências.

Ele não aprende representações mais gerais da linguagem.

---

#### O que queremos?

Queremos um sistema capaz de aprender sozinho.

Ao invés de armazenar contagens diretamente, queremos armazenar parâmetros ajustáveis.

Esses parâmetros serão modificados durante o treinamento.

É exatamente isso que uma rede neural faz.

In [4]:
# ==================================================
# SEÇÃO 2 - ESCALA
# ==================================================

vocab_exemplo = 50_000

possiveis_transicoes = vocab_exemplo * vocab_exemplo

print(
    f"Possíveis transições: {possiveis_transicoes:,}"
)

Possíveis transições: 2,500,000,000


#### Exercício de Reflexão

Imagine um vocabulário com milhões de tokens.

Pergunta:

Memorizar todas as transições continua sendo uma boa estratégia?

Quais problemas poderiam surgir?

#### Conceito Importante

Grande parte da evolução da IA pode ser vista como uma tentativa de substituir tabelas explícitas por representações aprendidas automaticamente.

### 3. Introdução aos Parâmetros Treináveis

#### Objetivo

Compreender a principal diferença entre um modelo clássico e uma rede neural.

No modelo anterior armazenávamos contagens.

Agora armazenaremos pesos.

---

#### O que é um peso?

Um peso é simplesmente um número ajustável.

Durante o treinamento esses números serão modificados.

O objetivo é fazer com que o modelo produza previsões cada vez melhores.

---

#### Uma analogia

Imagine uma mesa de som.

Cada botão pode aumentar ou diminuir um aspecto do áudio.

Uma rede neural funciona de maneira parecida.

O treinamento ajusta milhares ou milhões desses "botões" automaticamente.

In [5]:
# ==================================================
# SEÇÃO 3 - PESOS
# ==================================================

import torch

peso = torch.tensor(0.5)

print("Peso inicial:")
print(peso)

Peso inicial:
tensor(0.5000)


#### Exercício de Reflexão

Se um peso influencia uma previsão incorreta, o que deveria acontecer com ele?

Ele deveria permanecer igual ou ser ajustado?

Por quê?

#### Conceito Importante

Aprender significa ajustar parâmetros.

Toda rede neural, independentemente do tamanho, aprende modificando pesos ao longo do treinamento.

### 4. Preparando o Ambiente Neural

#### Objetivo

Antes de criarmos nossa primeira rede neural, precisamos reconstruir algumas estruturas básicas utilizadas no notebook anterior.

Isso permitirá que este notebook seja executado de forma independente.

Além disso, prepararemos os dados para utilização com PyTorch.

---

#### O que vamos criar?

Precisamos de:

- Dataset
- Vocabulário
- Tokenização
- Sequência Numérica
- Tensor PyTorch

Esses elementos serão utilizados durante todo o treinamento.

---

#### Por que estamos repetindo isso?

No notebook anterior utilizávamos listas Python comuns.

Agora entraremos no mundo do Deep Learning.

Para isso precisaremos representar nossos dados utilizando tensores do PyTorch.

In [11]:
# ==================================================
# SEÇÃO 4 - SETUP
# ==================================================

import torch

with open("../../data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print(f"Dataset size: {len(data)}")
print(f"Vocabulary size: {vocab_size}")

Dataset size: 52
Vocabulary size: 16


#### Exercício de Reflexão

Observe o tipo da variável `data`.

Pergunta:

Por que estamos utilizando um tensor do PyTorch em vez de uma lista Python comum?

O que uma biblioteca de Deep Learning pode oferecer além do simples armazenamento de números?

#### Conceito Importante

Deep Learning é construído sobre operações matemáticas realizadas em tensores.

Um tensor pode ser visto como uma generalização de:

- Escalares
- Vetores
- Matrizes

Praticamente tudo que uma rede neural faz consiste em transformar tensores em outros tensores.

### 5. Criando Nossa Primeira Matriz de Pesos

#### Objetivo

No notebook anterior utilizávamos uma tabela de contagens.

Agora vamos substituí-la por uma matriz de parâmetros treináveis.

Esses parâmetros serão ajustados automaticamente durante o treinamento.

---

#### O que é uma matriz de pesos?

Imagine uma tabela onde:

- Cada linha representa um token atual.
- Cada coluna representa um possível próximo token.

Cada posição contém um número chamado peso.

Inicialmente esses valores são aleatórios.

Durante o treinamento eles serão ajustados para representar padrões da linguagem.

---

#### Comparação

Modelo Clássico:

Token Atual
↓
Tabela de Probabilidades
↓
Próximo Token

Modelo Neural:

Token Atual
↓
Matriz de Pesos
↓
Probabilidades
↓
Próximo Token

In [12]:
# ==================================================
# SEÇÃO 5 - MATRIZ DE PESOS
# ==================================================

weights = torch.randn(
    (vocab_size, vocab_size),
    requires_grad=True
)

print(weights.shape)

torch.Size([16, 16])


In [13]:
weights[:5, :5]

tensor([[ 0.3442,  0.7805,  1.1382, -0.8356,  0.4255],
        [-1.6860,  0.9794,  0.8988, -2.1434,  0.3008],
        [ 2.8523,  0.9400,  1.7041,  1.5311, -1.4797],
        [ 0.1823,  2.0051,  0.3200,  0.6443, -1.3387],
        [-0.5621,  0.3364,  0.3363, -1.2470,  0.5173]],
       grad_fn=<SliceBackward0>)

#### Exercício de Reflexão

Observe os valores da matriz.

Pergunta:

Esses números possuem algum significado neste momento?

Se todos foram gerados aleatoriamente, como o modelo poderá aprender algo útil a partir deles?

#### Conceito Importante

No início do treinamento a rede não sabe absolutamente nada.

Os pesos são inicializados aleatoriamente.

O aprendizado consiste em modificar esses valores para que as previsões produzidas pela rede se aproximem cada vez mais dos dados observados.

#### Recapitulando

Até agora construímos:

Texto
↓
Tokenização
↓
Tensor
↓
Matriz de Pesos

Ainda não realizamos nenhuma previsão.

Na próxima seção veremos o que acontece quando um token entra na rede neural pela primeira vez.